<a href="https://colab.research.google.com/github/ananya2006sarkar/fashion-recommender/blob/main/fashion_recommender.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics.pairwise import cosine_similarity

In [12]:
df = pd.read_csv('/styles.csv', on_bad_lines='skip')
print(df.shape)
print(df.columns.tolist())
df.head()

(44424, 10)
['id', 'gender', 'masterCategory', 'subCategory', 'articleType', 'baseColour', 'season', 'year', 'usage', 'productDisplayName']


,id,gender,masterCategory,subCategory,articleType,baseColour,season,year,usage,productDisplayName
0,15970,Men,Apparel,Topwear,Shirts,Navy Blue,Fall,2011.0,Casual,Turtle Check Men Navy Blue Shirt
1,39386,Men,Apparel,Bottomwear,Jeans,Blue,Summer,2012.0,Casual,Peter England Men Party Blue Jeans
2,59263,Women,Accessories,Watches,Watches,Silver,Winter,2016.0,Casual,Titan Women Silver Watch
3,21379,Men,Apparel,Bottomwear,Track Pants,Black,Fall,2011.0,Casual,Manchester United Men Solid Black Track Pants
4,53759,Men,Apparel,Topwear,Tshirts,Grey,Summer,2012.0,Casual,Puma Men Grey T-shirt


In [13]:
df = df[df['masterCategory'] == 'Apparel'].copy()
df = df[['id', 'gender', 'subCategory', 'articleType', 'baseColour', 'usage', 'productDisplayName']]
df = df.dropna()
df = df.reset_index(drop=True)

print(df.shape)
print(df['gender'].unique())
print(df['usage'].unique())
print(df['subCategory'].unique())
df.head(10)

(21367, 7)
['Men' 'Women' 'Girls' 'Boys' 'Unisex']
['Casual' 'Ethnic' 'Formal' 'Sports' 'Smart Casual' 'Party' 'Travel']
['Topwear' 'Bottomwear' 'Innerwear' 'Saree' 'Dress'
 'Loungewear and Nightwear' 'Apparel Set' 'Socks']


,id,gender,subCategory,articleType,baseColour,usage,productDisplayName
0,15970,Men,Topwear,Shirts,Navy Blue,Casual,Turtle Check Men Navy Blue Shirt
1,39386,Men,Bottomwear,Jeans,Blue,Casual,Peter England Men Party Blue Jeans
2,21379,Men,Bottomwear,Track Pants,Black,Casual,Manchester United Men Solid Black Track Pants
3,53759,Men,Topwear,Tshirts,Grey,Casual,Puma Men Grey T-shirt
4,1855,Men,Topwear,Tshirts,Grey,Casual,Inkfruit Mens Chain Reaction T-shirt
5,30805,Men,Topwear,Shirts,Green,Ethnic,Fabindia Men Striped Green Shirt
6,26960,Women,Topwear,Shirts,Purple,Casual,Jealous 21 Women Purple Shirt
7,12369,Men,Topwear,Shirts,Purple,Formal,Reid & Taylor Men Check Purple Shirts
8,42419,Girls,Topwear,Tops,White,Casual,Gini and Jony Girls Knit White Top
9,51832,Women,Innerwear,Bra,Beige,Casual,Bwitch Beige Full-Coverage Bra BW335


In [14]:
le = LabelEncoder()
df['gender_enc']   = le.fit_transform(df['gender'])
df['subCat_enc']   = le.fit_transform(df['subCategory'])
df['article_enc']  = le.fit_transform(df['articleType'])
df['colour_enc']   = le.fit_transform(df['baseColour'])
df['usage_enc']    = le.fit_transform(df['usage'])

df[['productDisplayName','gender_enc','subCat_enc','article_enc','colour_enc','usage_enc']].head()

,productDisplayName,gender_enc,subCat_enc,article_enc,colour_enc,usage_enc
0,Turtle Check Men Navy Blue Shirt,2,7,38,22,0
1,Peter England Men Party Blue Jeans,2,1,16,2,0
2,Manchester United Men Solid Black Track Pants,2,1,50,1,0
3,Puma Men Grey T-shirt,2,7,54,11,0
4,Inkfruit Mens Chain Reaction T-shirt,2,7,54,11,0


In [15]:
def recommend(gender, subCategory, colour, usage, top_n=5):
    filtered = df[
        (df['gender'] == gender) &
        (df['subCategory'] == subCategory)
    ].copy()

    if filtered.empty:
        print("no products found!"); return

    filtered = filtered.drop_duplicates(subset='productDisplayName')
    filtered['score'] = 0
    filtered.loc[filtered['baseColour'] == colour, 'score'] += 2
    filtered.loc[filtered['usage'] == usage, 'score'] += 1

    results = filtered.sort_values('score', ascending=False).head(top_n)
    results = results[['productDisplayName', 'articleType', 'baseColour', 'usage', 'score']].reset_index(drop=True)
    results.index += 1

    print(f"top {top_n} recommendations:\n")
    return results

In [16]:
recommend(
    gender= "Women", subCategory = "Topwear", colour= "Blue", usage = "Casual",top_n= 5)

top 5 recommendations:



,productDisplayName,articleType,baseColour,usage,score
1,ADIDAS Women Blue TShirt,Tshirts,Blue,Casual,3
2,UCB Women's Sleeveless Cowl Neck Blue Top,Tops,Blue,Casual,3
3,Forever New Women Long Blue Jacket,Jackets,Blue,Casual,3
4,Scullers For Her Women Blue Top,Tops,Blue,Casual,3
5,Wrangler Women Ditsy Polka Blue Grey Shirt,Shirts,Blue,Casual,3
